# Streaming Products from S3 During Commissioning

## Introduction

This notebook demonstrates how to search MAST for commissioning-like data products made during MRT-7b, and then shows how to stream those data products from MAST into memory without the need to make local copies.

<div class="alert alert-warning" style="color:black; background-color:#82caff; border-color:blue;">
    <b>NOTE:</b> This notebook points you to the MAST Integration & Test (I&T) instance. MAST OPS has been cleared out in preparation for launch, but MAST I&T has retained a copy of selected datasets to aid in commissioning development. If you do not have access to MAST I&T, please contact <a href="mailto:archive@stsci.edu?subject=Roman MAST InT Access">archive@stsci.edu</a> to request access.</div>

This tutorial has been adapted mainly from the MAST notebook [MAST Metadata Search](https://github.com/spacetelescope/mast_notebooks/blob/roman-prelaunch/notebooks/Roman/MAST_metadata_search/MAST_metadata_search.ipynb) and the Nexus tutorial [Working with ADSF](https://github.com/spacetelescope/roman_notebooks/blob/main/notebooks/working_with_asdf/working_with_asdf.ipynb), with additional information added to support streaming from MAST search results.

## Imports

In [1]:
import os
import requests
import roman_datamodels as rdm
from astroquery.mast import MastMissions, Conf
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from dotenv import load_dotenv

First, we create our `MastMissions` object to act as our gateway to MAST for searches. If you do not have a MAST AUTH token for accessing MAST (needed for Roman OPS access until after commissioning), or if your token has expired, make one now by visiting [MAST.Auth](https://auth.mastint.stsci.edu/token?suggested_name=Astroquery&suggested_scope=mast:exclusive_access) for MAST I&T.

Once you have a token, you can set it up as an environment variable for later use, or you can store it in a file called `mast_api_token.txt`. If you use the file method, it must be in the same directory as this notebook or wherever you query MAST for data (both searches and retrievals). The environment variable is typically more convenient.

In [2]:
# Your .env file should look like this (no comment lines):
# MAST_API_TOKEN="your_token"

_ = load_dotenv('.env')  ## Change this to your .env file!

In [3]:
test_server = 'https://mastint.stsci.edu'
Conf.server = test_server
missions = MastMissions(mission='Roman')
missions._service_api_connection.MISSIONS_URL = test_server + '/search/'


In [4]:
# Login to search and retrieve Roman data
token = os.getenv("MAST_API_TOKEN")
if token is None:
    try:
        with open("mast_api_token.txt") as f:
            token = f.read().strip()
    except FileNotFoundError:
        raise ValueError("MAST token not found!")


In [7]:
with open("mast_api_token.txt") as f:
    token = f.read().strip()

In [5]:
token

'ad41e5519f024fdd94b8be8410821c49'

In [6]:
missions.login(token=token)


ConnectTimeout: HTTPConnectionPool(host='auth.mastint.stsci.edu', port=80): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='auth.mastint.stsci.edu', port=80) at 0x137ac27b0>, 'Connection to auth.mastint.stsci.edu timed out. (connect timeout=None)'))

In [2]:
# Create MastMissions object and assign mission to 'roman'

test_server = 'https://mastint.stsci.edu'
Conf.server = test_server
missions = MastMissions(mission='Roman')
missions._service_api_connection.MISSIONS_URL = test_server + '/search/'

# Login to search and retrieve Roman data
token = os.getenv("MAST_API_TOKEN")
if token is None:
    try:
        with open("mast_api_token.txt") as f:
            token = f.read().strip()
    except FileNotFoundError:
        raise ValueError("MAST token not found!")

missions.login(token=token)
               
print(f'Mission: {missions.mission}')
print(f'Service: {missions.service}')

ConnectTimeout: HTTPConnectionPool(host='auth.mastint.stsci.edu', port=80): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='auth.mastint.stsci.edu', port=80) at 0x12e9c14f0>, 'Connection to auth.mastint.stsci.edu timed out. (connect timeout=None)'))

Next, as an example, here we show how to search for data from Program ID 120. Be as specific as you can as the more results MAST returns the longer this cell will take to execute.

In [ ]:
# Create a dictionary of search criteria
search = {'program': 120, 'productLevel': 1}

# Query with column criteria
results = missions.query_criteria(**search)

# Display the first 5 results
print(f'Total number of results: {len(results)}')
results[:5]

## Streaming Files from MAST into Memory

Now that we have search results, we can select one and stream it into memory to examine it. You will primarily be interested in the Level 1 (L1) uncalibrated ramps or the Level 2 (L2) calibrated rate images, but there are some additional products as well. See the [WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products) article on RDox for more information.

For the retrieval below, the extension for L1 products is `uncal.asdf`, while for L2 products it is `cal.asdf`.

First, we get the list of data products (files) associated with the search results in the table above. We use the `get_unique_product_list()` method to retrieve only the unique data products.

In [ ]:
products = missions.get_unique_product_list(results[:5])
products[:5]

In [ ]:
filtered = missions.filter_products(products, file_suffix='_cal')
filtered

In [ ]:
tab_index = 3  # Choose which row from the filtered product table you want to retrieve
af = missions.read_product(filtered['filename'][tab_index])

Next we can transform the `AsdfFile` object to a datamodel. This is optional and comes down to how you prefer to interface with the data in the ASDF file.

In [ ]:
dm = rdm.open(af)

## Working with ASDF

For more information on working with ASDF files, see the Nexus tutorial [Working with ADSF](https://github.com/spacetelescope/roman_notebooks/blob/main/notebooks/working_with_asdf/working_with_asdf.ipynb). Briefly, you can access the metadata using dot notation as below:

In [ ]:
dm.meta.observation

From the `observation` section of the metadata, we can see that this does conform to our search for our MRT-7b example (we see it is program ID 114, pass 57, etc.). If we want to plot the data, which for MRT-7b is a test pattern, then we can also do that.

<div class="alert alert-warning" style="color:black; background-color:#ffc5c5; border-color:red;">
    <b>NOTE:</b> If you run the cell below, or in any other way try to access part of the file and get a long exception that ends with "ClientResponseError: 403, message='Forbidden'" followed by a long URI, then try to re-run the missions.read_product() command above. We believe this may be due to a timeout during the streaming, and are investigating.
</div>

In [ ]:
fig, ax = plt.subplots()
norm = simple_norm(dm.data, percent=99.9)
ax.imshow(dm.data, origin='lower', norm=norm)
ax.set_title(filtered[tab_index]['dataset'])
ax.set_ylabel('Science Y (pixels)')
ax.set_xlabel('Science X (pixels)');